In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path().resolve().parent
DATA_PATH = BASE_DIR / "data" / "cc_fraud_train_processed.csv"
OUTPUT_PATH = BASE_DIR / "data" / "risk_scored_data_v2_improved.csv"

# ------------------------
# 데이터 로딩
# ------------------------
df = pd.read_csv(DATA_PATH)
print("데이터 shape:", df.shape)

# ------------------------
# is_night 생성
# ------------------------
if "is_night" not in df.columns:
    df["is_night"] = df["trans_hour"].apply(
        lambda x: 1 if x < 6 or x >= 22 else 0
    )
    print("✅ is_night 생성 완료")

# ------------------------
# 1️⃣ 금액 점수 (구간화)
# ------------------------
df["amt_score"] = 0

df.loc[df["amt_zscore"] > 1.5, "amt_score"] = 10
df.loc[df["amt_zscore"] > 2.5, "amt_score"] = 25
df.loc[df["amt_zscore"] > 3.5, "amt_score"] = 40

# ------------------------
# 2️⃣ 시간 점수 (구간화)
# ------------------------
df["time_score"] = 0

df.loc[df["hour_dev"] > 1, "time_score"] = 10
df.loc[df["hour_dev"] > 2, "time_score"] = 20
df.loc[df["hour_dev"] > 3, "time_score"] = 30

# ------------------------
# 3️⃣ 고액 거래 (완화)
# ------------------------
df["high_amt_flag"] = (df["amt_zscore"] > 3).astype(int)

# 👉 기존 30 → 20으로 완화 (과탐지 감소)
df["high_amt_score"] = df["high_amt_flag"] * 20

# ------------------------
# 4️⃣ Interaction (최소 추가)
# ------------------------
df["interaction_score"] = 0

# 👉 핵심 조합 1개만 사용 (폭발 방지)
df.loc[
    (df["amt_zscore"] > 2.5) & (df["hour_dev"] > 2),
    "interaction_score"
] = 15

# ------------------------
# 5️⃣ Risk Score 계산 (100점 기준 유지)
# ------------------------
df["risk_score"] = (

    df["amt_score"]            # 최대 40
    + df["high_amt_score"]     # 최대 20
    + df["time_score"]         # 최대 30
    + df["is_night"] * 10      # 최대 10
    + df["interaction_score"]  # 최대 15

).clip(0, 100).round(3)

# ------------------------
# 결과 확인
# ------------------------
print("\n📊 risk_score 통계")
print(df["risk_score"].describe())

print("\n📊 risk_score 분포")
print(pd.cut(
    df["risk_score"],
    bins=[0,20,40,60,80,100]
).value_counts().sort_index())

# ------------------------
# 저장
# ------------------------
df.to_csv(OUTPUT_PATH, index=False)
print("✅ 저장 완료:", OUTPUT_PATH)

데이터 shape: (1296675, 22)
✅ is_night 생성 완료

📊 risk_score 통계
count    1.296675e+06
mean     2.800610e+01
std      1.513477e+01
min      0.000000e+00
25%      2.000000e+01
50%      3.000000e+01
75%      4.000000e+01
max      1.000000e+02
Name: risk_score, dtype: float64

📊 risk_score 분포
risk_score
(0, 20]      246983
(20, 40]     891230
(40, 60]       7460
(60, 80]       4091
(80, 100]     14541
Name: count, dtype: int64
✅ 저장 완료: C:\Users\kim\Desktop\2ndprj\fold\Teamproject (1)\data\risk_scored_data_v2_improved.csv
